# Colab Natural AI Pipeline

This notebook prepares a clean natural-language dataset, retrains the classifier, rebuilds the FAISS index without label leakage, and exports the artifacts you need back into the backend project.

## 1. Mount Drive And Install Dependencies

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q pandas tqdm datasets transformers torch faiss-cpu sentencepiece accelerate pydantic-settings

## 2. Set Paths

Update these paths to match your Drive layout.

In [ ]:
from pathlib import Path

REPO_DIR = Path('/content/GP_MainBranch-master')
DDX_DIR = Path('/content/drive/MyDrive/DDX/raw/ddxplus_hf')
EXPORT_DIR = Path('/content/drive/MyDrive/DDX/exports_natural_ai')
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

print('REPO_DIR =', REPO_DIR)
print('DDX_DIR =', DDX_DIR)
print('EXPORT_DIR =', EXPORT_DIR)

## 3. Verify Repo And Dataset

In [ ]:
assert REPO_DIR.exists(), f'Repo not found: {REPO_DIR}'
assert DDX_DIR.exists(), f'DDX dataset not found: {DDX_DIR}'

print('Repo exists:', REPO_DIR)
print('Dataset exists:', DDX_DIR)
print('Repo files sample:')
for p in list(REPO_DIR.iterdir())[:10]:
    print(' -', p.name)

## 4. Build Clean Natural CSVs

This step removes label leakage. The generated `combined_text` contains only demographics and symptoms.

In [ ]:
%cd /content/GP_MainBranch-master

!python backend/scripts/build_ddxplus_natural_csvs.py --dataset-dir "$DDX_DIR" --out-dir data/processed_ddxplus

## 5. Train Natural-Language Classifier

In [ ]:
%cd /content/GP_MainBranch-master

!python backend/scripts/train_clinicalbert_classifier.py \
  --train-csv data/processed_ddxplus/train_natural.csv \
  --val-csv data/processed_ddxplus/validate_natural.csv \
  --test-csv data/processed_ddxplus/test_natural.csv \
  --output-dir backend/artifacts/clinicalbert_classifier_natural \
  --epochs 3 \
  --batch-size 8 \
  --max-length 256

## 6. Rebuild FAISS From Natural Inputs

This rebuild uses only patient demographics and symptoms. It does not index the answer label inside the text.

In [ ]:
%cd /content/GP_MainBranch-master

!python backend/scripts/rebuild_faiss_from_ddx.py --dataset-dir "$DDX_DIR" --out-dir backend/faiss_data_natural

## 7. Export Artifacts Back To Drive

In [ ]:
import shutil
from pathlib import Path

src_classifier = Path('/content/GP_MainBranch-master/backend/artifacts/clinicalbert_classifier_natural')
src_faiss = Path('/content/GP_MainBranch-master/backend/faiss_data_natural')
src_processed = Path('/content/GP_MainBranch-master/data/processed_ddxplus')

dst_classifier = EXPORT_DIR / 'clinicalbert_classifier_natural'
dst_faiss = EXPORT_DIR / 'faiss_data_natural'
dst_processed = EXPORT_DIR / 'processed_ddxplus'

for dst in (dst_classifier, dst_faiss, dst_processed):
    if dst.exists():
        shutil.rmtree(dst)

shutil.copytree(src_classifier, dst_classifier)
shutil.copytree(src_faiss, dst_faiss)
shutil.copytree(src_processed, dst_processed)

print('Exported classifier to', dst_classifier)
print('Exported faiss index to', dst_faiss)
print('Exported processed csvs to', dst_processed)

## 8. What To Copy Back Into The Project

After Colab finishes, copy these folders into your local project:

- `backend/artifacts/clinicalbert_classifier_natural`
- `backend/faiss_data_natural`

Then we can switch the backend config to use these natural-language artifacts and rerun end-to-end evaluation locally.